In [5]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from unsloth import FastLanguageModel


# Where your saved LoRA adapter folder is (the one you downloaded & re-uploaded / copied)
ADAPTER_DIR = Path("/workspace/MentorApp/lora_adapter").resolve()

# Where to save the merged Hugging Face model (this folder will be large)
MERGED_DIR = Path("/workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16").resolve()
MERGED_DIR.mkdir(parents=True, exist_ok=True)

LLAMA_DIR  = Path("/workspace/MentorApp/llama.cpp").resolve()

print("ADAPTER_DIR exists:", ADAPTER_DIR.exists(), ADAPTER_DIR)
print("MERGED_DIR:", MERGED_DIR)


ADAPTER_DIR exists: True /workspace/MentorApp/lora_adapter
MERGED_DIR: /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16


In [ ]:
# IMPORTANT:
# Your adapters were trained via (Q)LoRA on a 4-bit base, but for merging + GGUF FP16 you should
# merge into the full-precision base model (same architecture/revision family).
BASE_MODEL_FULL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Use FP16 for a clean FP16 -> GGUF pipeline (bfloat16 is also fine, but FP16 is the usual target)
dtype = torch.float16

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_FULL, use_fast=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_FULL,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.56s/it]


ValueError: Can't find 'adapter_config.json' at '/workspace/MentorApp/outputs/lora_adapter'

In [6]:
# Attach LoRA adapter onto the full-precision base model
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

# Merge LoRA weights into base and remove adapter layers
merged_model = model.merge_and_unload()

# (Optional) ensure merged weights are saved as FP16
merged_model = merged_model.to(dtype)

# Save merged HF model + tokenizer
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

print("✅ Merged HF model saved to:", MERGED_DIR)
print("   Base used:", BASE_MODEL_FULL)

✅ Merged HF model saved to: /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16
   Base used: Qwen/Qwen2.5-Coder-7B-Instruct
